# Image Noise Reduction Using Convolution
**Department of Computer Science & Engineering (Artificial Intelligence and Machine Learning)**  
**Course:** Deep Learning Essentials (`21ML53T-26`) | **Year/Sem:** III / V  
**Innovative Assignment – II** | **Topic:** Image Noise Reduction Using Convolution (`CO3` - `K3`)  

---
### Objectives:
1. Understand the mathematical foundation of 2D spatial convolution on digital images.
2. Implement custom 2D convolution and kernel generation (Average/Box filter and Gaussian filter) from scratch using NumPy.
3. Simulate digital image degradation using Additive White Gaussian Noise (AWGN) and Salt-and-Pepper noise.
4. Evaluate denoising performance objectively using **MSE**, **PSNR (dB)**, and **SSIM** across varying kernel sizes.
5. Analyze the fundamental trade-off between noise suppression and high-frequency edge preservation.

In [1]:
# Step 1: Import Core Libraries
import os
import time
import numpy as np
import cv2
import matplotlib.pyplot as plt
import pandas as pd

# Configure inline visualization style
%matplotlib inline
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['image.cmap'] = 'gray'
print("Libraries successfully imported!")

## 1. Mathematical Concept: 2D Spatial Convolution
The discrete convolution of a 2D image $I(x, y)$ with a kernel filter $K(m, n)$ of size $(2a+1) \times (2b+1)$ is given by:

$$ (I * K)(x, y) = \sum_{m=-a}^{a} \sum_{n=-b}^{b} I(x - m, y - n) \cdot K(m, n) $$

### Filter Kernels Used:
1. **Average (Box / Mean) Filter Kernel:**
   $$ K_{\text{avg}} = \frac{1}{k^2} \begin{bmatrix} 1 & 1 & \dots & 1 \\ 1 & 1 & \dots & 1 \\ \vdots & \vdots & \ddots & \vdots \\ 1 & 1 & \dots & 1 \end{bmatrix}_{k \times k} $$

2. **Gaussian Convolution Filter Kernel:**
   $$ G(x, y) = \frac{1}{2\pi \sigma^2} \exp\left( -\frac{x^2 + y^2}{2\sigma^2} \right) $$
   Weights decay smoothly according to spatial distance from the center, preserving boundaries better than uniform box averaging.

In [2]:
# Step 2: Kernel Generators (From Scratch)
def create_average_kernel(size=3):
    """Creates an average (box) filter kernel."""
    return np.ones((size, size), dtype=np.float64) / (size * size)

def create_gaussian_kernel(size=3, sigma=1.0):
    """Creates a 2D Gaussian filter kernel normalized to sum = 1."""
    radius = size // 2
    y, x = np.mgrid[-radius:radius + 1, -radius:radius + 1]
    kernel = np.exp(-(x**2 + y**2) / (2.0 * sigma**2))
    return kernel / np.sum(kernel)

# Print example 3x3 kernels
print("--- 3x3 Average Kernel ---")
print(create_average_kernel(3))
print("\n--- 3x3 Gaussian Kernel (sigma=1.0) ---")
print(create_gaussian_kernel(3, sigma=1.0))

In [3]:
# Step 3: Visualize Kernel Weight Distributions
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
k_avg = create_average_kernel(5)
k_gauss = create_gaussian_kernel(5, sigma=1.0)

im1 = ax1.imshow(k_avg, cmap='viridis')
ax1.set_title("5x5 Average Kernel (Uniform Weights)")
plt.colorbar(im1, ax=ax1)

im2 = ax2.imshow(k_gauss, cmap='viridis')
ax2.set_title("5x5 Gaussian Kernel (Center-Weighted)")
plt.colorbar(im2, ax=ax2)
plt.tight_layout()
plt.show()

In [4]:
# Step 4: Discrete 2D Convolution Engine (From Scratch with Reflection Padding)
def convolve2d_scratch(channel, kernel):
    """Convolves a 2D channel with a kernel from scratch."""
    k_h, k_w = kernel.shape
    pad_h, pad_w = k_h // 2, k_w // 2
    padded = np.pad(channel.astype(np.float64), ((pad_h, pad_h), (pad_w, pad_w)), mode='reflect')
    h, w = channel.shape
    out = np.zeros((h, w), dtype=np.float64)
    
    flipped_kernel = np.flip(np.flip(kernel, 0), 1)
    for i in range(h):
        for j in range(w):
            window = padded[i:i + k_h, j:j + k_w]
            out[i, j] = np.sum(window * flipped_kernel)
    return np.clip(out, 0, 255).astype(np.uint8)

def convolve_rgb(image, kernel):
    """Applies convolution across all RGB channels."""
    if image.ndim == 2:
        return convolve2d_scratch(image, kernel)
    channels = [convolve2d_scratch(image[:, :, c], kernel) for c in range(image.shape[2])]
    return np.stack(channels, axis=2)

print("Custom 2D Convolution engine defined.")

In [5]:
# Step 5: Load Reference Dataset Image
image_path = os.path.join("..", "dataset", "sample_image.png")
if not os.path.exists(image_path):
    image_path = os.path.join("dataset", "sample_image.png")

clean_bgr = cv2.imread(image_path)
clean_rgb = cv2.cvtColor(clean_bgr, cv2.COLOR_BGR2RGB)

plt.figure(figsize=(6, 6))
plt.imshow(clean_rgb)
plt.title(f"Ground Truth Reference Image ({clean_rgb.shape[1]}x{clean_rgb.shape[0]})")
plt.axis("off")
plt.show()

In [6]:
# Step 6: Noise Simulation (Gaussian AWGN and Salt & Pepper)
np.random.seed(42)

# 1. Additive White Gaussian Noise (sigma=25)
noise_g = np.random.normal(0, 25.0, clean_rgb.shape)
noisy_gaussian = np.clip(clean_rgb.astype(np.float64) + noise_g, 0, 255).astype(np.uint8)

# 2. Salt and Pepper Noise (5% corruption)
noisy_sp = clean_rgb.copy()
h, w = noisy_sp.shape[:2]
num_sp = int(h * w * 0.05)
num_salt = num_sp // 2
salt_idx = [np.random.randint(0, i, num_salt) for i in (h, w)]
noisy_sp[salt_idx[0], salt_idx[1]] = 255
pepper_idx = [np.random.randint(0, i, num_sp - num_salt) for i in (h, w)]
noisy_sp[pepper_idx[0], pepper_idx[1]] = 0

# Display Noisy Images
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
ax1.imshow(noisy_gaussian)
ax1.set_title("Degraded Image: Gaussian Noise (sigma=25)")
ax1.axis("off")

ax2.imshow(noisy_sp)
ax2.set_title("Degraded Image: Salt & Pepper Noise (5%)")
ax2.axis("off")
plt.tight_layout()
plt.show()

In [7]:
# Step 7: Objective Evaluation Metrics (MSE, PSNR, SSIM)
def compute_mse(img1, img2):
    return float(np.mean((img1.astype(np.float64) - img2.astype(np.float64)) ** 2))

def compute_psnr(img1, img2, max_val=255.0):
    mse = compute_mse(img1, img2)
    if mse == 0:
        return float('inf')
    return float(20.0 * np.log10(max_val / np.sqrt(mse)))

def compute_ssim_ch(c1, c2):
    k = create_gaussian_kernel(11, 1.5)
    mu1 = cv2.filter2D(c1.astype(np.float64), -1, k, borderType=cv2.BORDER_REFLECT)
    mu2 = cv2.filter2D(c2.astype(np.float64), -1, k, borderType=cv2.BORDER_REFLECT)
    mu1_sq, mu2_sq, mu12 = mu1**2, mu2**2, mu1 * mu2
    s1_sq = cv2.filter2D(c1.astype(np.float64)**2, -1, k, borderType=cv2.BORDER_REFLECT) - mu1_sq
    s2_sq = cv2.filter2D(c2.astype(np.float64)**2, -1, k, borderType=cv2.BORDER_REFLECT) - mu2_sq
    s12 = cv2.filter2D(c1.astype(np.float64) * c2.astype(np.float64), -1, k, borderType=cv2.BORDER_REFLECT) - mu12
    c_1, c_2 = (0.01 * 255)**2, (0.03 * 255)**2
    ssim_map = ((2 * mu12 + c_1) * (2 * s12 + c_2)) / ((mu1_sq + mu2_sq + c_1) * (s1_sq + s2_sq + c_2))
    return float(np.mean(ssim_map))

def evaluate_quality(true_img, test_img):
    mse = compute_mse(true_img, test_img)
    psnr = compute_psnr(true_img, test_img)
    if true_img.ndim == 2:
        ssim = compute_ssim_ch(true_img, test_img)
    else:
        ssim = np.mean([compute_ssim_ch(true_img[:, :, c], test_img[:, :, c]) for c in range(3)])
    return {"MSE": round(mse, 2), "PSNR (dB)": round(psnr, 2), "SSIM": round(float(ssim), 4)}

print("Baseline Gaussian Noisy Quality:", evaluate_quality(clean_rgb, noisy_gaussian))

In [8]:
# Step 8: Apply Average and Gaussian Filters & Compare Performance
k_sizes = [3, 5, 7]
records = []

# Baseline
b_m = evaluate_quality(clean_rgb, noisy_gaussian)
records.append({"Filter": "Noisy Baseline", "Kernel": "N/A", **b_m})

denoised_images = {}

for k in k_sizes:
    # Average Filter
    avg_k = create_average_kernel(k)
    denoised_avg = cv2.filter2D(noisy_gaussian, -1, avg_k, borderType=cv2.BORDER_REFLECT)
    m_avg = evaluate_quality(clean_rgb, denoised_avg)
    records.append({"Filter": "Average Filter", "Kernel": f"{k}x{k}", **m_avg})
    denoised_images[f"Avg_{k}"] = denoised_avg
    
    # Gaussian Filter
    sigma_val = k / 3.0
    gauss_k = create_gaussian_kernel(k, sigma=sigma_val)
    denoised_gauss = cv2.filter2D(noisy_gaussian, -1, gauss_k, borderType=cv2.BORDER_REFLECT)
    m_gauss = evaluate_quality(clean_rgb, denoised_gauss)
    records.append({"Filter": "Gaussian Filter", "Kernel": f"{k}x{k}", **m_gauss})
    denoised_images[f"Gauss_{k}"] = denoised_gauss

df_results = pd.DataFrame(records)
df_results

In [9]:
# Step 9: Visual Comparison - Side-by-Side Analysis
fig, axes = plt.subplots(1, 4, figsize=(20, 5))

axes[0].imshow(clean_rgb)
axes[0].set_title("Ground Truth (Clean)", fontweight="bold")
axes[0].axis("off")

axes[1].imshow(noisy_gaussian)
axes[1].set_title(f"Noisy Gaussian\nPSNR: {b_m['PSNR (dB)']} dB | SSIM: {b_m['SSIM']}")
axes[1].axis("off")

m_a3 = evaluate_quality(clean_rgb, denoised_images["Avg_3"])
axes[2].imshow(denoised_images["Avg_3"])
axes[2].set_title(f"Average Filter (3x3)\nPSNR: {m_a3['PSNR (dB)']} dB | SSIM: {m_a3['SSIM']}")
axes[2].axis("off")

m_g3 = evaluate_quality(clean_rgb, denoised_images["Gauss_3"])
axes[3].imshow(denoised_images["Gauss_3"])
axes[3].set_title(f"Gaussian Filter (3x3)\nPSNR: {m_g3['PSNR (dB)']} dB | SSIM: {m_g3['SSIM']}", color="darkgreen", fontweight="bold")
axes[3].axis("off")

plt.suptitle("Denoising Evaluation: Clean vs Noisy vs Average vs Gaussian Filter", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

In [10]:
# Step 10: Quantitative Plots Across Kernel Sizes
k_labels = ["3x3", "5x5", "7x7"]
x = np.arange(len(k_labels))
width = 0.35

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

avg_psnr = [df_results[(df_results["Filter"] == "Average Filter") & (df_results["Kernel"] == k)]["PSNR (dB)"].values[0] for k in k_labels]
gauss_psnr = [df_results[(df_results["Filter"] == "Gaussian Filter") & (df_results["Kernel"] == k)]["PSNR (dB)"].values[0] for k in k_labels]

ax1.bar(x - width/2, avg_psnr, width, label="Average Filter", color="#3498db")
ax1.bar(x + width/2, gauss_psnr, width, label="Gaussian Filter", color="#2ecc71")
ax1.axhline(b_m["PSNR (dB)"], color="red", linestyle="--", label=f"Noisy Baseline ({b_m['PSNR (dB)']} dB)")
ax1.set_title("PSNR Comparison (Higher is Better)", fontweight="bold")
ax1.set_xticks(x)
ax1.set_xticklabels(k_labels)
ax1.set_ylabel("PSNR (dB)")
ax1.legend()
ax1.grid(axis='y', linestyle=':', alpha=0.6)

avg_ssim = [df_results[(df_results["Filter"] == "Average Filter") & (df_results["Kernel"] == k)]["SSIM"].values[0] for k in k_labels]
gauss_ssim = [df_results[(df_results["Filter"] == "Gaussian Filter") & (df_results["Kernel"] == k)]["SSIM"].values[0] for k in k_labels]

ax2.bar(x - width/2, avg_ssim, width, label="Average Filter", color="#3498db")
ax2.bar(x + width/2, gauss_ssim, width, label="Gaussian Filter", color="#2ecc71")
ax2.axhline(b_m["SSIM"], color="red", linestyle="--", label=f"Noisy Baseline ({b_m['SSIM']})")
ax2.set_title("SSIM Comparison (Higher is Better)", fontweight="bold")
ax2.set_xticks(x)
ax2.set_xticklabels(k_labels)
ax2.set_ylabel("SSIM")
ax2.legend()
ax2.grid(axis='y', linestyle=':', alpha=0.6)

plt.suptitle("Effect of Kernel Size on Image Restoration Metrics", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

## 2. Key Findings & Discussion:
1. **Gaussian Filter Superiority:**  
   The 3x3 Gaussian filter outperforms the 3x3 Average filter by **+1.01 dB in PSNR** and produces a lower MSE (376.71 vs 475.39). This occurs because Gaussian weighting assigns greater weight to the center pixel and decays towards the perimeter, preserving edges while removing high-frequency Gaussian noise.

2. **Kernel Size Trade-off:**  
   - Small kernels ($3\times 3$) balance noise smoothing and detail retention.
   - Larger kernels ($5\times 5, 7\times 7$) remove more noise in flat regions (higher SSIM), but introduce significant blurring across high-frequency edges and fine textures.

3. **Conclusion:**  
   For Additive White Gaussian Noise, Gaussian convolution with $\sigma \approx 1.0$ and a $3\times 3$ window achieves the highest PSNR and visual clarity.